In [ ]:
import pandas as pd
import datetime
from bs4 import BeautifulSoup
import re
import selenium
import ast
import time

from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.webdriver.support.ui import Select
from selenium.webdriver.firefox.options import Options

pd.set_option('display.max_rows', 200)
pd.set_option('display.float_format', '{:,.2f}'.format)
pd.set_option("display.max_colwidth", None)
pd.set_option('display.width', 2000)
pd.set_option('display.max_columns', 200)

In [ ]:
options = Options()
options.add_experimental_option("debuggerAddress", "127.0.0.1:9222")
driver = webdriver.Chrome(options=options)

In [ ]:
resultats = []

for page in range(1, 88):

    driver.get(f"https://www.consilium.europa.eu/en/general-secretariat/corporate-policies/transparency/voting-results/?ActType=0&ActNumber=&DocNumber=&InterinstNumber=&Title=&DateFrom=&DateTo=&Session=0&CouncilAction=0&VotingRule=0&VotingProcedure=0&PolicyArea=0&Country=0&VoteFor=on&VoteAgainst=on&VoteAbstain=on&VoteNotParticipating=on&page={page}")
    print(f"Page {page}/87")
    
    WebDriverWait(driver, 60).until(
        EC.visibility_of_element_located((By.CSS_SELECTOR, "section.gsc-search-results__acts"))
    )

    try:
        element = WebDriverWait(driver, 5).until(
            EC.element_to_be_clickable((By.CSS_SELECTOR, "button.gsc-btn.gsc-btn--confirm"))
        )
        element.click()
    except:
        pass
    
    soup = BeautifulSoup(driver.page_source, "html.parser")

    bloc_1 = soup.find_all("strong", class_="gsc-title")
    bloc_2 = soup.find_all("ul", class_="gsc-search-results__act-meta")
    bloc_3 = soup.find_all("ul", class_="gsc-search-results__results")
    bloc_4 = soup.find_all("div", class_="gsc-search-results__status")
    bloc_votes = soup.find_all("section", class_="gsc-search-results__votes")

    for titre, meta1, meta2, result_vote, detail_votes in zip(bloc_1, bloc_2, bloc_3, bloc_4, bloc_votes):
        
        nom_loi = titre.text.strip()

        dico_meta1 = {}
        for li in meta1.find_all("li"):
            split = li.text.strip().split(":", 1)
            cle = split[0].strip()
            valeur = split[1].strip()
            dico_meta1[cle] = valeur

        lignes_meta2 = meta2.find_all("li")
        infos_meta2 = [li.text.strip() for li in lignes_meta2]

        statut = result_vote.find("strong").text.strip()

        votes = detail_votes.find_all("div", class_="gsc-votes__countries")
        pays_pour = [pays.text for pays in votes[0].find_all("abbr")]
        pays_contre = [pays.text for pays in votes[1].find_all("abbr")]
        pays_abstention = [pays.text for pays in votes[2].find_all("abbr")]
        pays_non_participation_vote = [pays.text for pays in votes[3].find_all("abbr")]

        texte_vote_conseil = {
            "nom_texte": nom_loi,

            "Act no (OJ)" : dico_meta1.get("Act no (OJ)", ""),
            "Doc. no.": dico_meta1.get("Doc. no.", ""),
            "Interinstitutional number": dico_meta1.get("Interinstitutional number", ""),
            "Date": dico_meta1.get("Date", ""),

            "Council": infos_meta2[0],
            "Policy area" : infos_meta2[1],
            "Legislative procedure": infos_meta2[2],
            "Voting rule": infos_meta2[3],
            "Action by the Council": infos_meta2[4],
            "Type of act": infos_meta2[5],

            "resultat_vote": statut,

            "pays_pour": pays_pour,
            "pays_contre": pays_contre,
            "pays_abstention": pays_abstention,
            "pays_non_participation_vote": pays_non_participation_vote
        }

        resultats.append(texte_vote_conseil)

    time.sleep(3)

In [ ]:
df = pd.DataFrame(resultats)

In [ ]:
df.to_csv("../data/raw/votes_conseil_ue_brut.csv", index=False)

In [ ]:
df["pays_pour"] = df["pays_pour"].apply(ast.literal_eval)
df["pays_contre"] = df["pays_contre"].apply(ast.literal_eval)
df["pays_abstention"] = df["pays_abstention"].apply(ast.literal_eval)
df["pays_non_participation_vote"] = df["pays_non_participation_vote"].apply(ast.literal_eval)

In [ ]:
df["Policy area"] = df["Policy area"].str.replace("Policy area: \n", "")
df["Council"] = df["Council"].str.replace("Council: \n", "")
df["Legislative procedure"] = df["Legislative procedure"].str.replace("Legislative procedure: \n", "")
df["Voting rule"] = df["Voting rule"].str.replace("Voting rule: \n", "")
df["Action by the Council"] = df["Action by the Council"].str.replace("Action by the Council: \n", "")
df["Type of act"] = df["Type of act"].str.replace("Type of act: \n", "")

In [ ]:
df["Date"] = pd.to_datetime(df["Date"], format="%d/%m/%Y")

In [ ]:
liste_votes = []

for index, row in df.iterrows():
    
    vote_pays = {}

    for pays in row["pays_pour"]:
        vote_pays[pays] = "pour"

    for pays in row["pays_contre"]:
        vote_pays[pays] = "contre"    

    for pays in row["pays_abstention"]:
        vote_pays[pays] = "abstention"

    for pays in row["pays_non_participation_vote"]:
        vote_pays[pays] = "non-participation au vote"
    
    liste_votes.append(vote_pays)

df_votes = pd.DataFrame(liste_votes)

In [ ]:
df_merged = pd.concat([df, df_votes], axis=1)

In [ ]:
df_merged["nom_texte"] = df_merged["nom_texte"].str.replace("- Non-approval of the European Parliament's amendment(s)", "")
df_merged["nom_texte"] = df_merged["nom_texte"].str.replace("Non-approval of the European Parliament's amendment(s)", "")

In [ ]:
df_merged["nom_texte"] = df_merged["nom_texte"].str.replace("\r\n", " ")

In [ ]:
df_merged = df_merged[df_merged["Date"] >= "2010-05-29"]

In [ ]:
classement_contre = df_merged.apply(lambda col: (col == "contre").sum()).sort_values(ascending=False)
classement_contre = classement_contre[classement_contre > 0]

In [ ]:
infog1 = pd.DataFrame({"contre": classement_contre})
infog1 = infog1.sort_values("contre", ascending=False)

In [ ]:
mapping_pays_datawrapper = {
    "AT": ":at: Autriche",
    "BE": ":be: Belgique",
    "BG": ":bg: Bulgarie",
    "CY": ":cy: Chypre",
    "CZ": ":cz: Tchéquie",
    "DE": ":de: Allemagne",
    "DK": ":dk: Danemark",
    "EE": ":ee: Estonie",
    "EL": ":gr: Grèce",
    "ES": ":es: Espagne",
    "FI": ":fi: Finlande",
    "FR": ":fr: France",
    "HR": ":hr: Croatie",
    "HU": ":hu: Hongrie",
    "IE": ":ie: Irlande",
    "IT": ":it: Italie",
    "LT": ":lt: Lituanie",
    "LU": ":lu: Luxembourg",
    "LV": ":lv: Lettonie",
    "MT": ":mt: Malte",
    "NL": ":nl: Pays-Bas",
    "PL": ":pl: Pologne",
    "PT": ":pt: Portugal",
    "RO": ":ro: Roumanie",
    "SE": ":se: Suède",
    "SI": ":si: Slovénie",
    "SK": ":sk: Slovaquie",
    "UK": ":gb: Royaume-Uni"
}

In [ ]:
infog1.index = infog1.index.map(mapping_pays_datawrapper)
infog1.to_csv("../data/clean/infog1_votes_contre_abstention.csv")

In [ ]:
infog2 = df_merged[df_merged["HU"] == "contre"].groupby(df_merged["Date"].dt.year).size()
infog2 = infog2.reindex(range(2010, 2027), fill_value=0)
infog2.to_csv("../data/clean/infog2_hu_contre_par_annee.csv")

In [ ]:
hu_contre_policy = df_merged[df_merged["HU"] == "contre"]["Policy area"].value_counts()
hu_abstention_policy = df_merged[df_merged["HU"] == "abstention"]["Policy area"].value_counts()
infog3 = pd.DataFrame({"contre": hu_contre_policy, "abstention": hu_abstention_policy}).fillna(0)
infog3 = infog3.astype(int)

In [ ]:
mapping_policy = {
    "Agriculture": "Agriculture",
    "Economy": "Économie",
    "Employment": "Emploi",
    "Energy": "Énergie",
    "Environment": "Environnement",
    "Finances": "Finances",
    "Foreign affairs": "Affaires étrangères",
    "Health": "Santé",
    "Institutional": "Institutionnel",
    "Internal market": "Marché intérieur",
    "Justice and Home Affairs": "Justice et affaires intérieures",
    "Social policy": "Politique sociale",
    "Telecommunications": "Télécommunications",
    "Transport": "Transport"
}

infog3.index = infog3.index.map(mapping_policy)

In [ ]:
infog3.to_csv("../data/clean/infog3_hu_contre_thematique.csv")

In [ ]:
infog4 = df_merged[df_merged["HU"] == "contre"]["pays_contre"].apply(len).value_counts().sort_index()

infog4.index = infog4.index.map({
    1: "Seule",
    2: "Avec 1 autre pays",
    3: "Avec 2 autres pays",
    4: "Avec 3 autres pays",
    5: "Avec 4 autres pays",
    6: "Avec 5 autres pays",
    7: "Avec 6 autres pays",
    9: "Avec 8 autres pays"
})

infog4.to_csv("../data/clean/infog4_hu_isolement.csv")

In [ ]:
infog5 = df_merged[df_merged["HU"] == "contre"]["pays_contre"].explode().value_counts()
infog5 = infog5[infog5.index != "HU"]

infog5.index = infog5.index.map(mapping_pays_datawrapper)

infog5.to_csv("../data/clean/infog5_hu_allies_contre.csv")

In [ ]:
df_conseil_europeen = pd.read_csv("../data/raw/veto_conseil_europeen_raw.csv", sep=";")

In [ ]:
df_conseil_europeen = df_conseil_europeen.rename(columns={
   "ms_veto": "pays_veto",
   "issue": "cause_du_veto",
   "issue_country": "pays_source_veto"
})

df_conseil_europeen = df_conseil_europeen.drop("Unnamed: 5", axis=1)

In [ ]:
df_conseil_europeen["date_veto"] = pd.to_datetime(df_conseil_europeen["date_veto"], format="%Y-%m-%d")
df_conseil_europeen = df_conseil_europeen[df_conseil_europeen["date_veto"] >= "2010-05-29"]
df_conseil_europeen = df_conseil_europeen.sort_values("date_veto", ascending=False)

In [ ]:
mapping_pays_conseil_europeen = {
    "AT": "Autriche",
    "BE": "Belgique",
    "BG": "Bulgarie",
    "CY": "Chypre",
    "CZ": "Tchéquie",
    "DE": "Allemagne",
    "DK": "Danemark",
    "EE": "Estonie",
    "GR": "Grèce",
    "ES": "Espagne",
    "FI": "Finlande",
    "FR": "France",
    "HR": "Croatie",
    "HU": "Hongrie",
    "IE": "Irlande",
    "IT": "Italie",
    "LT": "Lituanie",
    "LU": "Luxembourg",
    "LV": "Lettonie",
    "MT": "Malte",
    "NL": "Pays-Bas",
    "PL": "Pologne",
    "PT": "Portugal",
    "RO": "Roumanie",
    "SE": "Suède",
    "SI": "Slovénie",
    "SK": "Slovaquie",
    "GB": "Royaume-Uni",
    "UA": "Ukraine",
    "IL": "Israël",
    "CN": "Chine",
    "BY": "Biélorussie",
    "RU": "Russie",
    "AL": "Albanie",
    "MK": "Macédoine du Nord",
    "VE": "Venezuela",
    "GE": "Géorgie",
    "BG;RO": "Bulgarie et Roumanie"
}

In [ ]:
df_conseil_europeen["pays_veto"] = df_conseil_europeen["pays_veto"].map(mapping_pays_conseil_europeen)
df_conseil_europeen["pays_source_veto"] = df_conseil_europeen["pays_source_veto"].map(mapping_pays_conseil_europeen)

In [ ]:
mapping_causes_veto_conseil_europeen = {
    "climate": "Climat",
    "monetary": "Monétaire",
    "tax": "Taxes",
    "betting": "Paris sportifs",
    "peace": "Paix",
    "human rights": "Droits de l'homme",
    "embassy": "Ambassade",
    "migration": "Migrations",
    "accession": "Processus d'adhésion à l'UE",
    "settlements": "Colonisation israélienne en Cisjordanie",
    "ceasefire": "Cessez-le-feu entre Israël et la Palestine",
    "aid": "Aide financière à l'Ukraine",
    "media": "Médias",
    "budget": "Budget",
    "sanctions": "Sanctions",
    "invasion": "Invasion de l'Ukraine par la Russie",
    "schengen": "Schengen"
}

In [ ]:
mapping_grandes_categories = {
    "Climat": "Environnement",
    "Monétaire": "Économie",
    "Taxes": "Économie",
    "Budget": "Économie",
    "Paris sportifs": "Autre",
    "Paix": "International",
    "Droits de l'homme": "International",
    "Ambassade": "International",
    "Cessez-le-feu entre Israël et la Palestine": "International",
    "Aide financière à l'Ukraine": "International",
    "Sanctions": "International",
    "Invasion de l'Ukraine par la Russie": "International",
    "Colonisation israélienne en Cisjordanie": "International",
    "Migrations": "Migrations",
    "Schengen": "Institutionnel",
    "Processus d'adhésion à l'UE": "Institutionnel",
    "Médias": "Autre"
}

In [ ]:
df_conseil_europeen["cause_du_veto"] = df_conseil_europeen["cause_du_veto"].map(mapping_causes_veto_conseil_europeen)
df_conseil_europeen["thematique_du_veto"] = df_conseil_europeen["cause_du_veto"].map(mapping_grandes_categories)

In [ ]:
df_conseil_europeen.to_excel("../data/raw/vetos_conseil_europeen_semi_clean.xlsx")

# le DataFrame a été retravaillé sur Excel afin d'arriver à la version définitive de l'infographie